# 13 Skill Gap Engine & Career Readiness
**Enterprise HR AI — Workforce Intelligence & Upskilling Platform**

### Purpose:
Compute individual employee skill gaps (Required Skills - Current Skills) and calculate Career Progression Readiness %.


In [2]:
import os
import pandas as pd
import numpy as np

DATA_PROCESSED = "../data/processed"
df_attrition = pd.read_csv(os.path.join(DATA_PROCESSED, "employee_attrition_processed.csv"))
df_profiles = pd.read_csv(os.path.join(DATA_PROCESSED, "role_competency_profiles.csv"))
df_emp_skills = pd.read_csv(os.path.join(DATA_PROCESSED, "employee_skills_controlled.csv"))

role_skills_map = {row['job_role']: set(row['required_skills_list'].split('|')) for _, row in df_profiles.iterrows()}
emp_skills_map = df_emp_skills.groupby('employee_id')['skill_name'].apply(set).to_dict()

gap_records = []

for _, emp in df_attrition.iterrows():
    emp_id = emp['employee_id']
    role = emp['job_role']
    
    required = role_skills_map.get(role, set())
    current = emp_skills_map.get(emp_id, set())
    
    missing = required - current
    matched = required.intersection(current)
    
    total_req = len(required) if len(required) > 0 else 1
    readiness_pct = np.round((len(matched) / total_req) * 100, 2)
    
    gap_records.append({
        'employee_id': emp_id,
        'job_role': role,
        'total_required_skills': len(required),
        'current_skills_count': len(current),
        'matched_skills_count': len(matched),
        'missing_skills_count': len(missing),
        'career_readiness_pct': readiness_pct,
        'missing_skills_list': "|".join(sorted(missing)),
        'current_skills_list': "|".join(sorted(current))
    })

df_gaps = pd.DataFrame(gap_records)
out_gaps = os.path.join(DATA_PROCESSED, "employee_skill_gaps.csv")
df_gaps.to_csv(out_gaps, index=False)

print("=== EMPLOYEE SKILL GAPS & READINESS (Sample 5) ===")
print(df_gaps[['employee_id', 'job_role', 'current_skills_count', 'missing_skills_count', 'career_readiness_pct']].head().to_string(index=False))


=== EMPLOYEE SKILL GAPS & READINESS (Sample 5) ===
 employee_id              job_role  current_skills_count  missing_skills_count  career_readiness_pct
           1       Sales Executive                     9                    10                 47.37
           2    Research Scientist                    11                    11                 50.00
           4 Laboratory Technician                     9                    11                 45.00
           5    Research Scientist                    10                    12                 45.45
           7 Laboratory Technician                     9                    11                 45.00
